# STM32F429 ADC (Analog-to-Digital Converter) Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using the ADC peripheral on STM32F429 microcontrollers. We'll cover everything from basic analog-to-digital conversion concepts to advanced features like DMA and multi-channel reading.

## Table of Contents
1. [Introduction to ADC](#introduction)
2. [STM32F429 ADC Overview](#adc-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic ADC Reading](#basic-reading)
6. [Voltage Measurements](#voltage-measurement)
7. [Configuration Options](#configuration)
8. [Multi-Channel Reading](#multi-channel)
9. [Continuous Conversion](#continuous)
10. [DMA Operations](#dma)
11. [Internal Sensors](#internal-sensors)
12. [Advanced Features](#advanced)
13. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to ADC

An Analog-to-Digital Converter (ADC) converts continuous analog signals into discrete digital values. This is essential for interfacing with sensors that produce analog outputs.

### Key Concepts:
- **Analog Signal**: Continuous voltage signal (0-3.3V)
- **Digital Value**: Discrete numerical representation
- **Resolution**: Number of bits (6, 8, 10, 12 bits)
- **Reference Voltage**: Maximum measurable voltage (3.3V)
- **Sampling Rate**: How fast conversions occur
- **Quantization**: Process of converting analog to digital

### ADC Formula:
```
Digital Value = (Analog Voltage × Max Digital Value) / Reference Voltage

For 12-bit ADC:
Digital Value = (Analog Voltage × 4095) / 3.3V
```

### Applications:
- Sensor reading (temperature, pressure, light)
- Audio processing
- Voltage monitoring
- Touch sensing
- Battery level monitoring
- Signal analysis

<a id='adc-overview'></a>
## 2. STM32F429 ADC Overview

The STM32F429 features three 12-bit ADCs with multiple channels and advanced features.

### Key Specifications:
- **Resolution**: 6-bit, 8-bit, 10-bit, or 12-bit
- **Channels**: 16 external + 3 internal (temperature, Vref, Vbat)
- **Conversion Time**: 0.4µs (2.4MSPS) at 12-bit resolution
- **Input Range**: 0V to VREF+ (typically 3.3V)
- **Trigger Sources**: Software, timers, external signals
- **Operating Modes**: Single, continuous, scan, discontinuous
- **DMA Support**: Direct memory access for high-speed transfers

### ADC Features:
- **Multi-channel**: Read multiple analog inputs
- **Internal sensors**: Temperature, reference voltage, battery voltage
- **Flexible triggering**: Software or hardware triggered
- **DMA capability**: High-speed data transfer
- **Interrupt support**: Conversion complete notifications
- **Calibration**: Built-in calibration for accuracy
- **Oversampling**: Improved resolution through averaging

### Digital Value Ranges:
- **12-bit**: 0 to 4095 (most common)
- **10-bit**: 0 to 1023
- **8-bit**: 0 to 255
- **6-bit**: 0 to 63

### Voltage Conversion:
```
Voltage = (Digital Value × Reference Voltage) / Max Digital Value
Voltage = (Digital Value × 3.3V) / 4095  (for 12-bit)
```

### ADC Register Map (Reference Manual)

The STM32F429 ADC peripheral uses several registers for configuration and operation:

| Register | Address Offset | Description |
|----------|----------------|-------------|
| ADC_SR   | 0x00 | Status Register |
| ADC_CR1  | 0x04 | Control Register 1 |
| ADC_CR2  | 0x08 | Control Register 2 |
| ADC_SMPR1 | 0x0C | Sample Time Register 1 |
| ADC_SMPR2 | 0x10 | Sample Time Register 2 |
| ADC_JOFR1-4 | 0x14-0x20 | Injected Channel Offset Registers |
| ADC_HTR  | 0x24 | Watchdog Higher Threshold Register |
| ADC_LTR  | 0x28 | Watchdog Lower Threshold Register |
| ADC_SQR1 | 0x2C | Regular Sequence Register 1 |
| ADC_SQR2 | 0x30 | Regular Sequence Register 2 |
| ADC_SQR3 | 0x34 | Regular Sequence Register 3 |
| ADC_JSQR | 0x38 | Injected Sequence Register |
| ADC_JDR1-4 | 0x3C-0x48 | Injected Data Registers |
| ADC_DR   | 0x4C | Regular Data Register |

### Key Register Configurations:

#### ADC_CR1 (Control Register 1):
- **Bits 26-24**: RES[2:0] - Resolution selection
  - 00: 12-bit (15 ADCCLK cycles)
  - 01: 10-bit (13 ADCCLK cycles)
  - 10: 8-bit (11 ADCCLK cycles)
  - 11: 6-bit (9 ADCCLK cycles)
- **Bit 23**: AWDEN - Analog watchdog enable
- **Bit 22**: JAWDEN - Injected analog watchdog enable
- **Bits 19-16**: DISCNUM[2:0] - Discontinuous mode channel count
- **Bit 14**: JDISCEN - Discontinuous mode on injected channels
- **Bit 13**: DISCEN - Discontinuous mode on regular channels
- **Bit 12**: JAUTO - Automatic injected group conversion
- **Bit 11**: AWDSGL - Enable watchdog on single channel
- **Bit 10**: SCAN - Scan mode enable
- **Bit 9**: JEOCIE - Interrupt enable for injected channels
- **Bit 8**: AWDIE - Analog watchdog interrupt enable
- **Bit 7**: EOCIE - End of conversion interrupt enable
- **Bits 4-0**: AWDCH[4:0] - Analog watchdog channel select

#### ADC_CR2 (Control Register 2):
- **Bit 30**: SWSTART - Start conversion of regular channels
- **Bit 26**: EXTEN[1:0] - External trigger enable
- **Bits 27-24**: EXTSEL[3:0] - External event select
- **Bit 22**: JSWSTART - Start conversion of injected channels
- **Bit 21**: JEXTEN[1:0] - Injected external trigger enable
- **Bits 20-16**: JEXTSEL[4:0] - Injected external event select
- **Bit 11**: ALIGN - Data alignment (0=right, 1=left)
- **Bit 10**: EOCS - End of conversion selection
- **Bit 9**: DDS - DMA disable selection
- **Bit 8**: DMA - Direct memory access mode
- **Bit 3**: RSTCAL - Reset calibration
- **Bit 2**: CAL - Calibration
- **Bit 1**: CONT - Continuous conversion
- **Bit 0**: ADON - A/D converter ON/OFF

#### ADC_SMPR1 & ADC_SMPR2 (Sample Time Registers):
- Control sampling time for each channel
- **000**: 3 cycles, **001**: 15 cycles, **010**: 28 cycles
- **011**: 56 cycles, **100**: 84 cycles, **101**: 112 cycles
- **110**: 144 cycles, **111**: 480 cycles

#### ADC_SQR1, ADC_SQR2, ADC_SQR3 (Regular Sequence Registers):
- Define conversion sequence for regular channels
- **ADC_SQR1[23:20]**: L[3:0] - Regular channel sequence length
- **ADC_SQR1[19:15]**: SQ16 - 16th conversion in regular sequence
- Each channel occupies 5 bits in sequence registers

### ADC Clock Configuration:
- **Maximum ADC Clock**: 36MHz
- **Prescaler Options**: PCLK2/2, PCLK2/4, PCLK2/6, PCLK2/8
- **Conversion Time**: 12.5 + sampling cycles
- **Total Conversion Time**: Varies with resolution and sampling time

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- Analog sensors or voltage sources
- Connecting wires
- Optional: Potentiometer, temperature sensor, etc.

### ADC Channel Pin Mapping:

| Channel | Pin | Description | ADC Instance |
|---------|-----|-------------|--------------|
| ADC_CHANNEL_0 | PA0 | Analog Input 0 | ADC1 |
| ADC_CHANNEL_1 | PA1 | Analog Input 1 | ADC1 |
| ADC_CHANNEL_2 | PA2 | Analog Input 2 | ADC1 |
| ADC_CHANNEL_3 | PA3 | Analog Input 3 | ADC1 |
| ADC_CHANNEL_4 | PA4 | Analog Input 4 | ADC1 |
| ADC_CHANNEL_5 | PA5 | Analog Input 5 | ADC1 |
| ADC_CHANNEL_6 | PA6 | Analog Input 6 | ADC1 |
| ADC_CHANNEL_7 | PA7 | Analog Input 7 | ADC1 |
| ADC_CHANNEL_8 | PB0 | Analog Input 8 | ADC1 |
| ADC_CHANNEL_9 | PB1 | Analog Input 9 | ADC1 |
| ADC_CHANNEL_10 | PC0 | Analog Input 10 | ADC1 |
| ADC_CHANNEL_11 | PC1 | Analog Input 11 | ADC1 |
| ADC_CHANNEL_12 | PC2 | Analog Input 12 | ADC1 |
| ADC_CHANNEL_13 | PC3 | Analog Input 13 | ADC1 |
| ADC_CHANNEL_14 | PC4 | Analog Input 14 | ADC1 |
| ADC_CHANNEL_15 | PC5 | Analog Input 15 | ADC1 |
| ADC_CHANNEL_TEMP | Internal | Temperature Sensor | ADC1 |
| ADC_CHANNEL_VREF | Internal | Voltage Reference | ADC1 |
| ADC_CHANNEL_VBAT | Internal | Battery Voltage | ADC1 |

### Hardware Considerations:
- **Input Voltage Range**: 0V to 3.3V maximum
- **Input Impedance**: High impedance inputs
- **Sampling Time**: Affects conversion accuracy
- **Reference Voltage**: 3.3V (VDDA)
- **Power Supply**: Clean, stable power required

### Circuit Examples:
```
Potentiometer Voltage Divider:
3.3V ────[10KΩ]─────── ADC Pin
                    │
                 [POT]
                    │
                   GND

Sensor with Voltage Divider:
Sensor ────[R1]─────── ADC Pin
       │         │
       R2        C1 (0.1µF)
       │         │
      GND       GND
```

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Files:
- `adc.h` - Header file with function prototypes
- `adc.c` - Main implementation
- STM32F4xx HAL Library (`stm32f4xx_hal.h`)
- Standard C libraries: `stdint.h`, `stdbool.h`, `string.h`

### Include Path Setup:
```c
#include "adc.h"
#include "stm32f4xx.h"  // HAL library
```

### Build Configuration:
- Add source files to project
- Include header paths
- Enable ADC peripheral in STM32CubeMX
- Configure ADC clock (max 36MHz)
- Enable GPIO ports for ADC pins

### Global ADC Handle:
```c
// Pre-defined ADC1 handle (most commonly used)
extern ADC_HandleStruct hadc1;
```

### Error Handling:
- All functions return `HAL_StatusTypeDef`
- Check return values: `HAL_OK`, `HAL_ERROR`, `HAL_BUSY`, `HAL_TIMEOUT`
- Use `ADC_GetStatusString()` for readable error messages

<a id='basic-reading'></a>
## 5. Basic ADC Reading

### ADC Configuration Structure:
```c
ADC_ConfigTypeDef adc_config = {
    .channel = ADC_CHANNEL_0,           // Channel to read
    .resolution = ADC_RESOLUTION_12B,   // 12-bit resolution
    .sampling_time = ADC_SAMPLETIME_84CYCLES, // 84 ADC cycles
    .conv_mode = ADC_MODE_SINGLE,       // Single conversion
    .dma_enabled = false                // No DMA
};
```

### Basic Initialization and Reading:
```c
#include "adc.h"

int main(void) {
    // Configure ADC
    ADC_ConfigTypeDef config = {
        .channel = ADC_CHANNEL_0_PA0,
        .resolution = ADC_RESOLUTION_12BIT,
        .sampling_time = ADC_SAMPLING_84_CYCLES,
        .conv_mode = ADC_MODE_SINGLE,
        .dma_enabled = false
    };
    
    // Initialize ADC
    HAL_StatusTypeDef status = ADC_Init(&hadc1, &config);
    if (status != HAL_OK) {
        printf("ADC initialization failed: %s\n", 
               ADC_GetStatusString(status));
        return -1;
    }
    
    // Read ADC value
    uint32_t raw_value;
    status = ADC_ReadChannel(&hadc1, ADC_CHANNEL_0_PA0, &raw_value);
    if (status == HAL_OK) {
        printf("Raw ADC Value: %lu\n", raw_value);
    }
    
    return 0;
}
```

### Step-by-Step Reading Process:
```c
// 1. Configure channel
ADC_ConfigChannel(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLING_84_CYCLES);

// 2. Start conversion
ADC_StartConversion(&hadc1);

// 3. Wait for completion
ADC_PollForConversion(&hadc1, 1000); // 1 second timeout

// 4. Get result
uint32_t value;
ADC_GetValue(&hadc1, &value);
```

### Understanding ADC Values:
- **12-bit ADC**: 0-4095 range
- **10-bit ADC**: 0-1023 range
- **8-bit ADC**: 0-255 range
- **6-bit ADC**: 0-63 range
- **0**: Represents 0V (GND)
- **Max Value**: Represents 3.3V (VREF+)
- **Mid Value**: Represents ~1.65V

<a id='voltage-measurement'></a>
## 6. Voltage Measurements

### Direct Voltage Reading (Recommended):
```c
#include "adc.h"

// Read voltage directly (most convenient)
float voltage = ADC_ReadChannelVoltage(&hadc1, ADC_CHANNEL_0_PA0);
if (voltage >= 0.0f) {
    printf("Voltage: %.3f V\n", voltage);
} else {
    printf("Error reading voltage\n");
}
```

### Manual Conversion:
```c
// Read raw value first
uint32_t raw_value;
ADC_ReadChannel(&hadc1, ADC_CHANNEL_0_PA0, &raw_value);

// Convert to voltage
float voltage = ADC_RawToVoltage(raw_value, ADC_RESOLUTION_12BIT);
printf("Voltage: %.3f V\n", voltage);

// Or use the macro
float voltage2 = ADC_RAW_TO_VOLTAGE(raw_value, ADC_RESOLUTION_12BIT);
```

### Voltage to Raw Conversion:
```c
// Convert voltage back to raw ADC value
uint32_t raw_from_voltage = ADC_VoltageToRaw(1.65f, ADC_RESOLUTION_12BIT);
// Should return 2048 (half of 4095)

// Or use the macro
uint32_t raw_macro = ADC_VOLTAGE_TO_RAW(1.65f, ADC_RESOLUTION_12BIT);
```

### Voltage Measurement Examples:
```c
// Read multiple voltages
float voltages[3];
uint32_t channels[3] = {ADC_CHANNEL_0_PA0, ADC_CHANNEL_1_PA1, ADC_CHANNEL_2_PA2};

for (int i = 0; i < 3; i++) {
    voltages[i] = ADC_ReadChannelVoltage(&hadc1, channels[i]);
    if (voltages[i] >= 0.0f) {
        printf("Channel %d: %.3f V\n", i, voltages[i]);
    }
}
```

### Voltage Ranges:
- **0.000V**: ADC value = 0
- **1.650V**: ADC value = 2048 (12-bit)
- **3.300V**: ADC value = 4095 (12-bit)
- **Reference Voltage**: Always 3.3V for STM32F429

### Accuracy Considerations:
- **Resolution**: 12-bit = ~0.8mV steps
- **Linearity**: ±1 LSB typical
- **Offset**: ±2 LSB typical
- **Temperature**: Affects accuracy
- **Noise**: Can reduce effective resolution

<a id='configuration'></a>
## 7. Configuration Options

### Resolution Settings:
```c
// Available resolutions
ADC_SetResolution(&hadc1, ADC_RESOLUTION_12B);  // 0-4095 (default)
ADC_SetResolution(&hadc1, ADC_RESOLUTION_10B);  // 0-1023
ADC_SetResolution(&hadc1, ADC_RESOLUTION_8B);   // 0-255
ADC_SetResolution(&hadc1, ADC_RESOLUTION_6B);   // 0-63
```

### Sampling Time Settings:
```c
// Available sampling times (ADC clock cycles)
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_3CYCLES);   // Fastest
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_15CYCLES);  // Fast
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_28CYCLES);  // Medium
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_56CYCLES);  // Slow
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_84CYCLES);  // Default
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_112CYCLES); // Slower
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_144CYCLES); // Slowest
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_480CYCLES); // Most accurate
```

### Conversion Modes:
```c
// Single conversion (one sample per trigger)
ADC_ConfigTypeDef config = {
    .conv_mode = ADC_MODE_SINGLE,
    // ... other settings
};

// Continuous conversion (continuous sampling)
ADC_ConfigTypeDef config = {
    .conv_mode = ADC_MODE_CONTINUOUS,
    // ... other settings
};
```

### Configuration Trade-offs:
- **Higher Resolution**: Better accuracy, slower conversion
- **Longer Sampling Time**: Better accuracy, slower conversion
- **Continuous Mode**: Real-time data, higher CPU usage
- **Single Mode**: Lower power, manual triggering

### Dynamic Configuration:
```c
// Change resolution at runtime
ADC_SetResolution(&hadc1, ADC_RESOLUTION_10B);

// Change sampling time for specific channel
ADC_SetSamplingTime(&hadc1, ADC_CHANNEL_0_PA0, ADC_SAMPLETIME_144CYCLES);

// Reconfigure entire ADC
ADC_ConfigTypeDef new_config = {
    .channel = ADC_CHANNEL_1_PA1,
    .resolution = ADC_RESOLUTION_8B,
    .sampling_time = ADC_SAMPLETIME_28CYCLES,
    .conv_mode = ADC_MODE_SINGLE,
    .dma_enabled = false
};
ADC_Init(&hadc1, &new_config);
```

### Register-Level Configuration (Advanced)

For advanced users who need direct register access, here are the key register configurations that correspond to the HAL settings:

#### Resolution Setup (ADC_CR1):
```c
// Set 12-bit resolution (default)
ADC1->CR1 &= ~(ADC_CR1_RES);  // Clear RES bits
// RES[2:0] = 000 for 12-bit

// Set 10-bit resolution
ADC1->CR1 |= ADC_CR1_RES_0;   // RES[2:0] = 001

// Set 8-bit resolution
ADC1->CR1 |= ADC_CR1_RES_1;   // RES[2:0] = 010

// Set 6-bit resolution
ADC1->CR1 |= ADC_CR1_RES_1 | ADC_CR1_RES_0;  // RES[2:0] = 011
```

#### Sampling Time Setup (ADC_SMPR1/ADC_SMPR2):
```c
// Set sampling time for channel 0 (ADC_SMPR2)
// SMP0[2:0] = 000 (3 cycles) - Fastest
ADC1->SMPR2 &= ~(ADC_SMPR2_SMP0);

// SMP0[2:0] = 100 (84 cycles) - Default
ADC1->SMPR2 |= ADC_SMPR2_SMP0_2;

// SMP0[2:0] = 111 (480 cycles) - Most accurate
ADC1->SMPR2 |= ADC_SMPR2_SMP0;
```

#### Conversion Mode Setup (ADC_CR2):
```c
// Single conversion mode (default)
ADC1->CR2 &= ~(ADC_CR2_CONT);

// Continuous conversion mode
ADC1->CR2 |= ADC_CR2_CONT;
```

#### DMA Setup (ADC_CR2):
```c
// Enable DMA
ADC1->CR2 |= ADC_CR2_DMA;

// Disable DMA
ADC1->CR2 &= ~(ADC_CR2_DMA);
```

#### Channel Sequence Setup (ADC_SQR1/ADC_SQR2/ADC_SQR3):
```c
// Set single channel conversion (channel 0)
ADC1->SQR3 = 0;              // Clear sequence register
ADC1->SQR3 |= (0 << 0);      // SQ1 = channel 0
ADC1->SQR1 &= ~(ADC_SQR1_L); // L[3:0] = 0000 (1 conversion)

// Set 3-channel sequence: CH0, CH1, CH2
ADC1->SQR3 = 0;
ADC1->SQR3 |= (0 << 0);      // SQ1 = CH0
ADC1->SQR3 |= (1 << 5);      // SQ2 = CH1
ADC1->SQR3 |= (2 << 10);     // SQ3 = CH2
ADC1->SQR1 |= (2 << 20);     // L[3:0] = 0010 (3 conversions)
```

#### ADC Enable/Disable (ADC_CR2):
```c
// Enable ADC
ADC1->CR2 |= ADC_CR2_ADON;

// Disable ADC
ADC1->CR2 &= ~(ADC_CR2_ADON);
```

#### Starting Conversion (ADC_CR2):
```c
// Start regular conversion
ADC1->CR2 |= ADC_CR2_SWSTART;
```

#### Reading Results (ADC_DR):
```c
// Read conversion result
uint32_t result = ADC1->DR;
```

### Register Configuration Examples:

#### Basic Single-Channel Setup:
```c
// 1. Enable ADC clock (RCC register)
RCC->APB2ENR |= RCC_APB2ENR_ADC1EN;

// 2. Configure resolution (12-bit default)
ADC1->CR1 &= ~(ADC_CR1_RES);

// 3. Configure sampling time (84 cycles for channel 0)
ADC1->SMPR2 |= ADC_SMPR2_SMP0_2;

// 4. Configure channel sequence (single channel 0)
ADC1->SQR3 = 0;              // SQ1 = channel 0
ADC1->SQR1 &= ~(ADC_SQR1_L); // 1 conversion

// 5. Enable ADC
ADC1->CR2 |= ADC_CR2_ADON;

// 6. Wait for ADC ready (optional)
while (!(ADC1->SR & ADC_SR_ADONS));

// 7. Start conversion
ADC1->CR2 |= ADC_CR2_SWSTART;

// 8. Wait for conversion complete
while (!(ADC1->SR & ADC_SR_EOC));

// 9. Read result
uint32_t value = ADC1->DR;
```

#### Continuous Conversion with DMA:
```c
// 1. Enable ADC and DMA clocks
RCC->APB2ENR |= RCC_APB2ENR_ADC1EN;
RCC->AHB1ENR |= RCC_AHB1ENR_DMA2EN;

// 2. Configure ADC for continuous mode
ADC1->CR2 |= ADC_CR2_CONT;

// 3. Enable DMA
ADC1->CR2 |= ADC_CR2_DMA;

// 4. Configure DMA stream (DMA2_Stream0 for ADC1)
DMA2_Stream0->CR = 0;  // Reset
DMA2_Stream0->CR |= (0 << 25);     // Channel 0
DMA2_Stream0->CR |= DMA_SxCR_MINC; // Memory increment
DMA2_Stream0->CR |= DMA_SxCR_CIRC; // Circular mode
DMA2_Stream0->CR |= DMA_SxCR_MSIZE_0; // Memory size 16-bit
DMA2_Stream0->CR |= DMA_SxCR_PSIZE_0; // Peripheral size 16-bit

// 5. Set DMA addresses
DMA2_Stream0->PAR = (uint32_t)&ADC1->DR;  // Peripheral address
DMA2_Stream0->M0AR = (uint32_t)adc_buffer; // Memory address
DMA2_Stream0->NDTR = BUFFER_SIZE;         // Number of data

// 6. Enable DMA
DMA2_Stream0->CR |= DMA_SxCR_EN;

// 7. Enable ADC
ADC1->CR2 |= ADC_CR2_ADON;

// 8. Start conversion
ADC1->CR2 |= ADC_CR2_SWSTART;
```

### Important Notes:
- **Register Access**: Direct register manipulation bypasses HAL protection
- **Clock Setup**: Always enable ADC clock before register access
- **Calibration**: Consider running ADC calibration before use
- **Interrupts**: Configure NVIC for interrupt-driven operation
- **Power Management**: ADC consumes power even when not converting

<a id='multi-channel'></a>
## 8. Multi-Channel Reading

### Configure Multiple Channels:
```c
// Define channels and sampling times
uint32_t channels[] = {
    ADC_CHANNEL_0_PA0,
    ADC_CHANNEL_1_PA1,
    ADC_CHANNEL_2_PA2
};

uint32_t sampling_times[] = {
    ADC_SAMPLETIME_84CYCLES,
    ADC_SAMPLETIME_84CYCLES,
    ADC_SAMPLETIME_84CYCLES
};

// Configure multi-channel
HAL_StatusTypeDef status = ADC_ConfigMultiChannel(&hadc1, 
                                                  channels, 
                                                  sampling_times, 
                                                  3);
```

### Read Multiple Channels:
```c
// Read all configured channels
uint32_t values[3];
HAL_StatusTypeDef status = ADC_ReadMultiChannel(&hadc1, values, 3);

if (status == HAL_OK) {
    for (int i = 0; i < 3; i++) {
        printf("Channel %d: %lu\n", i, values[i]);
    }
}
```

### Multi-Channel Voltage Reading:
```c
// Read voltages from multiple channels
uint32_t channels[] = {ADC_CHANNEL_0_PA0, ADC_CHANNEL_1_PA1, ADC_CHANNEL_2_PA2};
float voltages[3];

for (int i = 0; i < 3; i++) {
    voltages[i] = ADC_ReadChannelVoltage(&hadc1, channels[i]);
    if (voltages[i] >= 0.0f) {
        printf("Channel %d: %.3f V\n", i, voltages[i]);
    }
}
```

### Channel Scanning Example:
```c
#define NUM_CHANNELS 8

uint32_t all_channels[NUM_CHANNELS] = {
    ADC_CHANNEL_0_PA0, ADC_CHANNEL_1_PA1, ADC_CHANNEL_2_PA2,
    ADC_CHANNEL_3_PA3, ADC_CHANNEL_4_PA4, ADC_CHANNEL_5_PA5,
    ADC_CHANNEL_6_PA6, ADC_CHANNEL_7_PA7
};

uint32_t channel_values[NUM_CHANNELS];

// Scan all channels
for (int i = 0; i < NUM_CHANNELS; i++) {
    ADC_ReadChannel(&hadc1, all_channels[i], &channel_values[i]);
    HAL_Delay(10); // Small delay between readings
}

// Display results
for (int i = 0; i < NUM_CHANNELS; i++) {
    float voltage = ADC_RawToVoltage(channel_values[i], ADC_RESOLUTION_12B);
    printf("PA%d: %.3f V (%lu)\n", i, voltage, channel_values[i]);
}
```

### Multi-Channel Considerations:
- **Channel Count**: Up to 16 external channels
- **Timing**: Each channel adds conversion time
- **Memory**: Need arrays for storing results
- **Synchronization**: All channels use same configuration
- **Performance**: More channels = slower total conversion time

<a id='continuous'></a>
## 9. Continuous Conversion

### Start Continuous Conversion:
```c
// Configure for continuous mode
ADC_ConfigTypeDef config = {
    .channel = ADC_CHANNEL_0_PA0,
    .resolution = ADC_RESOLUTION_12B,
    .sampling_time = ADC_SAMPLETIME_84CYCLES,
    .conv_mode = ADC_MODE_CONTINUOUS,  // Continuous mode
    .dma_enabled = false
};

ADC_Init(&hadc1, &config);

// Start continuous conversion
HAL_StatusTypeDef status = ADC_StartContinuousConversion(&hadc1);
```

### Read Continuous Data:
```c
// Read continuously in a loop
while (1) {
    uint32_t value;
    HAL_StatusTypeDef status = ADC_GetValue(&hadc1, &value);
    
    if (status == HAL_OK) {
        float voltage = ADC_RawToVoltage(value, ADC_RESOLUTION_12B);
        printf("Continuous: %.3f V\n", voltage);
    }
    
    HAL_Delay(100); // Read every 100ms
}
```

### Stop Continuous Conversion:
```c
// Stop continuous conversion
ADC_StopContinuousConversion(&hadc1);
```

### Continuous Mode Example:
```c
#include "adc.h"

void monitor_voltage(void) {
    // Configure for continuous monitoring
    ADC_ConfigTypeDef config = {
        .channel = ADC_CHANNEL_0_PA0,
        .resolution = ADC_RESOLUTION_12B,
        .sampling_time = ADC_SAMPLETIME_84CYCLES,
        .conv_mode = ADC_MODE_CONTINUOUS,
        .dma_enabled = false
    };
    
    ADC_Init(&hadc1, &config);
    ADC_StartContinuousConversion(&hadc1);
    
    // Monitor for 10 seconds
    uint32_t start_time = HAL_GetTick();
    while (HAL_GetTick() - start_time < 10000) {
        uint32_t value;
        if (ADC_GetValue(&hadc1, &value) == HAL_OK) {
            float voltage = ADC_RawToVoltage(value, ADC_RESOLUTION_12B);
            printf("%.3f V\r", voltage); // Carriage return for updating line
            fflush(stdout);
        }
        HAL_Delay(50);
    }
    
    ADC_StopContinuousConversion(&hadc1);
}
```

### Continuous Mode Considerations:
- **CPU Usage**: Higher than single conversions
- **Real-time**: Provides continuous data stream
- **Power**: Slightly higher power consumption
- **Timing**: Fixed sampling rate based on configuration
- **Interrupts**: Can use interrupts for efficiency

<a id='dma'></a>
## 10. DMA Operations

### DMA Configuration:
```c
// Configure ADC with DMA enabled
ADC_ConfigTypeDef config = {
    .channel = ADC_CHANNEL_0_PA0,
    .resolution = ADC_RESOLUTION_12B,
    .sampling_time = ADC_SAMPLETIME_84CYCLES,
    .conv_mode = ADC_MODE_CONTINUOUS,
    .dma_enabled = true  // Enable DMA
};

ADC_Init(&hadc1, &config);
```

### DMA Buffer Reading:
```c
#define BUFFER_SIZE 100
uint32_t adc_buffer[BUFFER_SIZE];

// Start DMA conversion
HAL_StatusTypeDef status = ADC_StartDMA(&hadc1, adc_buffer, BUFFER_SIZE);

if (status == HAL_OK) {
    // Wait for DMA transfer complete or timeout
    uint32_t timeout = HAL_GetTick() + 5000; // 5 second timeout
    while (!ADC_IsConversionComplete(&hadc1) && HAL_GetTick() < timeout) {
        // Wait for conversion
    }
    
    // Process buffer data
    for (int i = 0; i < BUFFER_SIZE; i++) {
        float voltage = ADC_RawToVoltage(adc_buffer[i], ADC_RESOLUTION_12B);
        printf("Sample %d: %.3f V\n", i, voltage);
    }
}

// Stop DMA
ADC_StopDMA(&hadc1);
```

### DMA with Continuous Conversion:
```c
// Circular DMA buffer for continuous sampling
#define CIRCULAR_BUFFER_SIZE 1000
uint32_t circular_buffer[CIRCULAR_BUFFER_SIZE];
volatile uint32_t buffer_index = 0;

void start_continuous_dma(void) {
    ADC_ConfigTypeDef config = {
        .channel = ADC_CHANNEL_0_PA0,
        .resolution = ADC_RESOLUTION_12B,
        .sampling_time = ADC_SAMPLETIME_84CYCLES,
        .conv_mode = ADC_MODE_CONTINUOUS,
        .dma_enabled = true
    };
    
    ADC_Init(&hadc1, &config);
    
    // Start circular DMA
    ADC_StartDMA(&hadc1, circular_buffer, CIRCULAR_BUFFER_SIZE);
}

void process_latest_samples(void) {
    // Process latest samples from circular buffer
    static uint32_t last_index = 0;
    
    while (last_index != buffer_index) {
        float voltage = ADC_RawToVoltage(circular_buffer[last_index], ADC_RESOLUTION_12B);
        // Process voltage data
        last_index = (last_index + 1) % CIRCULAR_BUFFER_SIZE;
    }
}
```

### DMA Advantages:
- **CPU Free**: DMA handles data transfer
- **High Speed**: Minimal CPU intervention
- **Large Buffers**: Handle thousands of samples
- **Real-time**: Continuous data acquisition
- **Efficiency**: Lower power for data transfer

### DMA Considerations:
- **Memory Usage**: Large buffers consume RAM
- **Setup Complexity**: More complex configuration
- **Interrupt Handling**: DMA completion interrupts
- **Buffer Management**: Circular vs linear buffers

<a id='internal-sensors'></a>
## 11. Internal Sensors

### Temperature Sensor:
```c
#include "adc.h"

// Read internal temperature sensor
float temperature = ADC_ReadTemperature(&hadc1);
if (temperature > ADC_ABSOLUTE_ZERO) {  // Check for valid reading
    printf("Temperature: %.1f °C\n", temperature);
} else {
    printf("Temperature reading failed\n");
}
```

### Voltage Reference:
```c
// Read internal voltage reference
float vref_int = ADC_ReadVrefInt(&hadc1);
if (vref_int > 0.0f) {
    printf("VREF Internal: %.3f V\n", vref_int);
    
    // Calculate actual VDD from VREF reading
    // VDD = 3.3V * (VREFINT_CAL / VREFINT_READ)
    float actual_vdd = 3.3f * (1.21f / vref_int);  // 1.21V is typical VREFINT
    printf("Actual VDD: %.3f V\n", actual_vdd);
}
```

### Battery Voltage:
```c
// Read battery voltage (VBAT pin)
float vbat = ADC_ReadVbat(&hadc1);
if (vbat > 0.0f) {
    printf("Battery Voltage: %.3f V\n", vbat);
}
```

### Internal Sensor Configuration:
```c
// Internal sensors require specific ADC configuration
ADC_ConfigTypeDef temp_config = {
    .channel = ADC_CHANNEL_TEMPSENSOR,  // Internal temperature
    .resolution = ADC_RESOLUTION_12B,
    .sampling_time = ADC_SAMPLETIME_480CYCLES, // Longer sampling for accuracy
    .conv_mode = ADC_MODE_SINGLE,
    .dma_enabled = false
};

ADC_Init(&hadc1, &temp_config);
```

### Temperature Sensor Details:
- **Range**: -40°C to 125°C
- **Accuracy**: ±1.5°C typical
- **Formula**: Temperature = (Vsense - V25) / Avg_Slope + 25°C
- **V25**: 0.76V at 25°C
- **Avg_Slope**: 2.5mV/°C

### Internal Sensor Applications:
- **Temperature Monitoring**: System temperature
- **Voltage Monitoring**: Power supply health
- **Battery Level**: Battery voltage monitoring
- **Calibration**: Reference for external measurements
- **Diagnostics**: System health monitoring

<a id='advanced'></a>
## 12. Advanced Features

### Interrupt-Driven Conversion:
```c
// Register callback functions
void conversion_complete_callback(ADC_HandleStruct* hadc, uint32_t value) {
    float voltage = ADC_RawToVoltage(value, ADC_RESOLUTION_12B);
    printf("Conversion complete: %.3f V\n", voltage);
}

void error_callback(ADC_HandleStruct* hadc) {
    printf("ADC conversion error\n");
}

// Register callbacks
ADC_RegisterConvCompleteCallback(&hadc1, conversion_complete_callback);
ADC_RegisterErrorCallback(&hadc1, error_callback);

// Start interrupt-driven conversion
ADC_ReadChannel_IT(&hadc1, ADC_CHANNEL_0_PA0);
```

### Calibration:
```c
// Calibrate ADC for better accuracy
HAL_StatusTypeDef status = ADC_Calibrate(&hadc1);
if (status == HAL_OK) {
    printf("ADC calibration successful\n");
} else {
    printf("ADC calibration failed\n");
}
```

### Status Checking:
```c
// Check ADC status
HAL_StatusTypeDef status = ADC_GetStatus(&hadc1);
printf("ADC Status: %s\n", ADC_GetStatusString(status));

// Check if ADC is ready
if (ADC_IsReady(&hadc1)) {
    printf("ADC is ready\n");
}

// Check if conversion is complete
if (ADC_IsConversionComplete(&hadc1)) {
    printf("Conversion complete\n");
}
```

### Error Handling:
```c
// Comprehensive error handling
HAL_StatusTypeDef status = ADC_ReadChannel(&hadc1, ADC_CHANNEL_0_PA0, &value);

switch (status) {
    case HAL_OK:
        printf("Success: %lu\n", value);
        break;
    case HAL_ERROR:
        printf("ADC Error\n");
        ADC_ErrorHandler(&hadc1);
        break;
    case HAL_BUSY:
        printf("ADC Busy\n");
        break;
    case HAL_TIMEOUT:
        printf("ADC Timeout\n");
        break;
    default:
        printf("Unknown ADC status\n");
        break;
}
```

### Utility Functions:
```c
// Get maximum value for resolution
uint32_t max_12bit = ADC_GetMaxValue(ADC_RESOLUTION_12B);  // Returns 4095
uint32_t max_10bit = ADC_GetMaxValue(ADC_RESOLUTION_10B);  // Returns 1023

// Get channel name
const char* name = ADC_GetChannelName(ADC_CHANNEL_0_PA0);  // Returns "PA0"
printf("Channel name: %s\n", name);

// Get status string
const char* status_str = ADC_GetStatusString(HAL_OK);  // Returns "OK"
```

### Performance Optimization:
- **Use DMA** for high-speed continuous conversion
- **Choose appropriate resolution** for your accuracy needs
- **Optimize sampling time** based on signal characteristics
- **Use interrupts** to avoid polling
- **Batch readings** for multi-channel applications
- **Calibrate regularly** for best accuracy

<a id='troubleshooting'></a>
## 13. Troubleshooting

### Common Issues and Solutions:

#### 1. ADC Initialization Fails
**Symptoms:** `ADC_Init()` returns error
**Possible Causes:**
- GPIO pins not configured correctly
- ADC clock not enabled
- Invalid configuration parameters
- Hardware conflicts
**Solutions:**
- Check GPIO configuration in STM32CubeMX
- Verify ADC clock settings
- Validate configuration parameters
- Check for pin conflicts

#### 2. Incorrect ADC Readings
**Symptoms:** Values don't match expected voltages
**Possible Causes:**
- Wrong reference voltage assumption
- Incorrect resolution setting
- Uncalibrated ADC
- Noise on input signal
**Solutions:**
- Verify reference voltage (3.3V)
- Check resolution setting
- Run ADC calibration
- Add input filtering/decoupling

#### 3. Noisy Measurements
**Symptoms:** ADC values fluctuate when input is stable
**Possible Causes:**
- Insufficient sampling time
- Power supply noise
- Input signal noise
- Ground loops
**Solutions:**
- Increase sampling time
- Add decoupling capacitors
- Use shielded cables
- Improve grounding

#### 4. Conversion Timeouts
**Symptoms:** `ADC_PollForConversion()` times out
**Possible Causes:**
- Too short timeout value
- ADC not properly configured
- Hardware fault
- Clock issues
**Solutions:**
- Increase timeout value
- Check ADC configuration
- Verify hardware connections
- Check ADC clock frequency

#### 5. DMA Issues
**Symptoms:** DMA transfers fail or incomplete
**Possible Causes:**
- DMA not enabled in configuration
- DMA channel conflicts
- Buffer alignment issues
- Incorrect buffer size
**Solutions:**
- Enable DMA in ADC config
- Check DMA channel allocation
- Ensure proper buffer alignment
- Verify buffer size calculations

### Debug Tips:
```c
// Check ADC status
HAL_StatusTypeDef status = ADC_GetStatus(&hadc1);
printf("ADC Status: %s\n", ADC_GetStatusString(status));

// Test basic functionality
uint32_t test_value;
status = ADC_ReadChannel(&hadc1, ADC_CHANNEL_0_PA0, &test_value);
if (status == HAL_OK) {
    printf("Basic read OK: %lu\n", test_value);
} else {
    printf("Basic read failed: %s\n", ADC_GetStatusString(status));
}

// Check voltage conversion
float test_voltage = ADC_ReadChannelVoltage(&hadc1, ADC_CHANNEL_0_PA0);
if (test_voltage >= 0.0f) {
    printf("Voltage read OK: %.3f V\n", test_voltage);
} else {
    printf("Voltage read failed\n");
}

// Test different channels
for (uint32_t ch = ADC_CHANNEL_0; ch <= ADC_CHANNEL_7; ch++) {
    float v = ADC_ReadChannelVoltage(&hadc1, ch);
    printf("Channel %lu: %.3f V\n", ch, v);
    HAL_Delay(100);
}
```

### Performance Tips:
- **Use appropriate resolution** for your application
- **Optimize sampling time** based on signal bandwidth
- **Use DMA** for high-speed applications
- **Batch channel readings** when possible
- **Calibrate** for best accuracy
- **Filter inputs** to reduce noise

### Hardware Considerations:
- **Input Voltage Range**: 0V to VREF+ (3.3V)
- **Input Impedance**: High impedance inputs
- **Decoupling**: Use 0.1µF capacitors near ADC pins
- **Shielding**: Shield analog signals from digital noise
- **Ground Planes**: Separate analog and digital grounds

## Complete Example Program

```c
#include "adc.h"
#include "stm32f4xx.h"
#include <stdio.h>

int main(void) {
    printf("STM32F429 ADC Demonstration\n\n");
    
    // Initialize ADC with default configuration
    ADC_ConfigTypeDef adc_config = {
        .channel = ADC_CHANNEL_0_PA0,
        .resolution = ADC_RESOLUTION_12B,
        .sampling_time = ADC_SAMPLETIME_84CYCLES,
        .conv_mode = ADC_MODE_SINGLE,
        .dma_enabled = false
    };
    
    HAL_StatusTypeDef status = ADC_Init(&hadc1, &adc_config);
    if (status != HAL_OK) {
        printf("ADC initialization failed: %s\n", ADC_GetStatusString(status));
        return -1;
    }
    printf("ADC initialized successfully\n\n");
    
    // Calibrate ADC
    status = ADC_Calibrate(&hadc1);
    if (status == HAL_OK) {
        printf("ADC calibration successful\n\n");
    }
    
    // Demonstrate basic reading
    printf("=== Basic ADC Reading ===\n");
    uint32_t raw_value;
    status = ADC_ReadChannel(&hadc1, ADC_CHANNEL_0_PA0, &raw_value);
    if (status == HAL_OK) {
        float voltage = ADC_RawToVoltage(raw_value, ADC_RESOLUTION_12B);
        printf("Raw Value: %lu\n", raw_value);
        printf("Voltage: %.3f V\n\n", voltage);
    }
    
    // Demonstrate voltage reading
    printf("=== Direct Voltage Reading ===\n");
    float direct_voltage = ADC_ReadChannelVoltage(&hadc1, ADC_CHANNEL_0_PA0);
    if (direct_voltage >= 0.0f) {
        printf("Direct Voltage: %.3f V\n\n", direct_voltage);
    }
    
    // Demonstrate multi-channel reading
    printf("=== Multi-Channel Reading ===\n");
    uint32_t channels[] = {ADC_CHANNEL_0_PA0, ADC_CHANNEL_1_PA1};
    for (int i = 0; i < 2; i++) {
        float v = ADC_ReadChannelVoltage(&hadc1, channels[i]);
        if (v >= 0.0f) {
            printf("Channel %d: %.3f V\n", i, v);
        }
        HAL_Delay(100);
    }
    printf("\n");
    
    // Demonstrate internal sensors
    printf("=== Internal Sensors ===\n");
    float temperature = ADC_ReadTemperature(&hadc1);
    if (temperature > -273.0f) {
        printf("Temperature: %.1f °C\n", temperature);
    }
    
    float vref = ADC_ReadVrefInt(&hadc1);
    if (vref > 0.0f) {
        printf("VREF Internal: %.3f V\n", vref);
    }
    
    printf("\nADC demonstration complete!\n");
    
    return 0;
}
```

## Summary

This tutorial covered:
- ADC fundamentals and STM32F429 ADC capabilities
- Hardware setup and pin configurations
- Basic and advanced ADC operations
- Voltage measurements and conversions
- Multi-channel and continuous conversion
- DMA operations for high-speed data
- Internal sensor reading (temperature, Vref, Vbat)
- Configuration options and optimization
- Troubleshooting common issues

### Key Takeaways:
1. **Always check return values** from ADC functions
2. **Calibrate regularly** for accurate measurements
3. **Choose appropriate settings** for your application needs
4. **Use DMA** for high-speed continuous conversion
5. **Handle errors gracefully** in production code
6. **Consider noise** and use proper filtering

### Next Steps:
- Experiment with different resolutions and sampling times
- Implement sensor data processing algorithms
- Add DMA for real-time data acquisition
- Integrate with other peripherals
- Develop application-specific ADC configurations

### Additional Resources:
- STM32F429 Reference Manual (ADC chapter)
- STM32F4 HAL Driver documentation
- Application notes on ADC best practices
- Sensor datasheets for proper interfacing

For more detailed information, refer to the STM32F429 datasheet and ADC application notes.